# 00-2. Preprocessing (CPTAC-PDAC)

In [1]:
library(data.table)
library(vespa)

In [2]:
if (!dir.exists("./data/cptac-pdac")) {
  out <- system2(
    "bash",
    "./tools/scripts/cptac-pdac.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

In [3]:
if (!file.exists("./tools/references/library.fasta")) {
  out <- system2(
    "bash",
    "./tools/scripts/fasta.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

## Import data

In [4]:
phospho <- fread("./data/cptac-pdac/phosphoproteomics_site_level_MD_abundance_tumor.cct")

In [5]:
phospho[, gene_id := Gene]

phospho[, protein_id := sub("_[STY][0-9]+$", "", Index)]
phospho[, phosphosite := sub("^.*_([STY][0-9]+)$", "\\1", Index)]

phospho[, modified_peptide_sequence := Peptide]
phospho[, peptide_sequence := toupper(Peptide)]

phospho[, site_id := paste(gene_id, protein_id, phosphosite, sep = ":")]
phospho[, peptide_id := paste(Index, Peptide, sep = "__")]

In [6]:
sample_cols <- setdiff(
  names(phospho),
  c(
    "Index",
    "Gene",
    "Peptide",
    "gene_id",
    "protein_id",
    "phosphosite",
    "modified_peptide_sequence",
    "peptide_sequence",
    "site_id",
    "peptide_id"
  )
)

phospho_long <- melt(
  phospho,
  id.vars = c(
    "gene_id",
    "protein_id",
    "peptide_id",
    "site_id",
    "modified_peptide_sequence",
    "peptide_sequence",
    "phosphosite"
  ),
  measure.vars = sample_cols,
  variable.name = "run_id",
  value.name = "peptide_intensity",
  variable.factor = FALSE,
  na.rm = TRUE
)

phospho_long[, run_id := as.character(run_id)]

phospho_long <- phospho_long[
  ,
  .(
    gene_id,
    protein_id,
    peptide_id,
    site_id,
    modified_peptide_sequence,
    peptide_sequence,
    phosphosite,
    run_id,
    peptide_intensity
  )
]

dim(phospho_long)
head(phospho_long)

[1] 2592677       9

gene_id,protein_id,peptide_id,site_id,modified_peptide_sequence,peptide_sequence,phosphosite,run_id,peptide_intensity
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
ACADVL,NP_000009.1,NP_000009.1_S489__ELSGLGsALK,ACADVL:NP_000009.1:S489,ELSGLGsALK,ELSGLGSALK,S489,C3N-03884,13.84222
PSEN1,NP_000012.1,NP_000012.1_S367__AAVQELSSsILAGEDPEER,PSEN1:NP_000012.1:S367,AAVQELSSsILAGEDPEER,AAVQELSSSILAGEDPEER,S367,C3N-03884,18.93493
ALDOA,NP_000025.1,NP_000025.1_S132__GVVPLAGTNGETTTQGLDGLsER,ALDOA:NP_000025.1:S132,GVVPLAGTNGETTTQGLDGLsER,GVVPLAGTNGETTTQGLDGLSER,S132,C3N-03884,17.19216
ALDOA,NP_000025.1,NP_000025.1_S276__TVPPAVTGITFLSGGQsEEEASINLNAINK,ALDOA:NP_000025.1:S276,TVPPAVTGITFLSGGQsEEEASINLNAINK,TVPPAVTGITFLSGGQSEEEASINLNAINK,S276,C3N-03884,16.82438
ALDOA,NP_000025.1,NP_000025.1_S36__GILAADEsTGSIAK,ALDOA:NP_000025.1:S36,GILAADEsTGSIAK,GILAADESTGSIAK,S36,C3N-03884,23.08933
ALDOA,NP_000025.1,NP_000025.1_S39__GILAADESTGsIAKR,ALDOA:NP_000025.1:S39,GILAADESTGsIAKR,GILAADESTGSIAKR,S39,C3N-03884,22.97128


In [7]:
proteo <- fread("./data/cptac-pdac/proteomics_gene_level_MD_abundance_tumor.cct")

In [8]:
setnames(proteo, names(proteo)[1], "gene_id")

proteo[, gene_id := trimws(gene_id)]

sample_cols <- setdiff(names(proteo), "gene_id")

obs_n <- rowSums(!is.na(proteo[, ..sample_cols]))

summary(obs_n)

proteo <- proteo[obs_n > 0]

dim(proteo)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0    70.0   135.0   105.5   140.0   140.0 

[1] 11631   141

In [9]:
fasta_headers <- readLines("tools/references/library.fasta")
fasta_headers <- fasta_headers[grepl("^>", fasta_headers)]

protein_id <- sub("^>[^|]*\\|([^|]+)\\|.*", "\\1", fasta_headers)

gene_symbol <- ifelse(
  grepl(" GN=", fasta_headers),
  sub(".* GN=([^ ]+).*", "\\1", fasta_headers),
  NA_character_
)

map_dt <- data.table(
  gene_id = gene_symbol,
  protein_id = protein_id,
  fasta_header = fasta_headers
)

map_dt <- map_dt[!is.na(gene_id)]

map_dt[, is_sp := grepl("^>sp\\|", fasta_header)]
setorder(map_dt, gene_id, -is_sp)

map_dt <- map_dt[, .SD[1], by = gene_id]
map_dt <- map_dt[, .(gene_id, protein_id)]

In [10]:
proteo_annot <- merge(
  proteo,
  map_dt,
  by = "gene_id",
  all.x = TRUE
)

cat("genes in proteo:", nrow(proteo_annot), "\n")
cat("mapped genes:", sum(!is.na(proteo_annot$protein_id)), "\n")
cat("unmapped genes:", sum(is.na(proteo_annot$protein_id)), "\n")

head(proteo_annot[is.na(protein_id), gene_id], 50)

genes in proteo: 11631 
mapped genes: 11480 
unmapped genes: 151 


[1] "AAED1"           "ADGRE5"          "ADSS"            "ADSSL1"         
 [5] "AES"             "ANKHD1-EIF4EBP3" "APOBEC3A_B"      "ATP5MF-PTCD1"   
 [9] "ATP5S"           "BCL2L2-PABPN1"   "BUB1B-PAK6"      "C10orf88"       
[13] "C11orf58"        "C12orf43"        "C12orf75"        "C15orf38-AP3S2" 
[17] "C15orf48"        "C16orf45"        "C17orf49"        "C17orf97"       
[21] "C19orf33"        "C19orf38"        "C19orf66"        "C19orf70"       
[25] "C1orf116"        "C1orf123"        "C1orf35"         "C21orf2"        
[29] "C2orf54"         "C3orf52"         "C3orf58"         "C4B_2"          
[33] "C5orf15"         "C6orf106"        "C6orf203"        "C6orf222"       
[37] "C6orf58"         "C7orf43"         "C7orf55-LUC7L2"  "C8orf59"        
[41] "CENPS-CORT"      "COL4A3BP"        "COMMD3-BMI1"     "CORO7-PAM16"    
[45] "CTGF"            "CXorf36"         "CYR61"           "DEFA1B"         
[49] "DIRC2"           "F8A2"

In [11]:
proteo_annot <- proteo_annot[!is.na(protein_id)]

sample_cols <- setdiff(names(proteo_annot), c("gene_id", "protein_id"))

proteo_long <- melt(
  proteo_annot,
  id.vars = c("gene_id", "protein_id"),
  measure.vars = sample_cols,
  variable.name = "run_id",
  value.name = "peptide_intensity",
  variable.factor = FALSE,
  na.rm = FALSE
)

proteo_long[, run_id := as.character(run_id)]

proteo_long[, peptide_id := protein_id]
proteo_long[, modified_peptide_sequence := protein_id]
proteo_long[, peptide_sequence := protein_id]
proteo_long[, phosphosite := "PA"]
proteo_long[, site_id := paste(gene_id, protein_id, phosphosite, sep = ":")]

proteo_long <- proteo_long[
  ,
  .(
    gene_id,
    protein_id,
    peptide_id,
    site_id,
    modified_peptide_sequence,
    peptide_sequence,
    phosphosite,
    run_id,
    peptide_intensity
  )
]

dim(proteo_long)
head(proteo_long)

[1] 1607200       9

gene_id,protein_id,peptide_id,site_id,modified_peptide_sequence,peptide_sequence,phosphosite,run_id,peptide_intensity
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
A1BG,P04217,P04217,A1BG:P04217:PA,P04217,P04217,PA,C3L-03394,28.67421
A1CF,Q9NQ94,Q9NQ94,A1CF:Q9NQ94:PA,Q9NQ94,Q9NQ94,PA,C3L-03394,24.02035
A2M,P01023,P01023,A2M:P01023:PA,P01023,P01023,PA,C3L-03394,29.77424
A2ML1,A8K2U0,A8K2U0,A2ML1:A8K2U0:PA,A8K2U0,A8K2U0,PA,C3L-03394,NA
A4GALT,Q9NPC4,Q9NPC4,A4GALT:Q9NPC4:PA,Q9NPC4,Q9NPC4,PA,C3L-03394,NA
A4GNT,Q9UNA3,Q9UNA3,A4GNT:Q9UNA3:PA,Q9UNA3,Q9UNA3,PA,C3L-03394,NA


In [12]:
dir.create("./data/cptac-pdac/processed")

In [13]:
saveRDS(phospho_long, "./data/cptac-pdac/processed/CPTAC_PDAC_phospho.rds")
saveRDS(proteo_long, "./data/cptac-pdac/processed/CPTAC_PDAC_proteo.rds")